# Fix Duplicates in Merged Crime Data

This notebook identifies and removes duplicate records from the merged dataset.

In [1]:
import pandas as pd
import numpy as np

# Load merged data
print("Loading merged crime data...")
df = pd.read_csv('../files/merged_crime_data_2003_2025.csv')
print(f"✓ Loaded {len(df):,} records")
print(f"\nColumns: {list(df.columns)}")

Loading merged crime data...
✓ Loaded 1,317,318 records

Columns: ['Incident_Number', 'Incident_Date', 'Incident_Time', 'Original_Category', 'Unified_Category', 'Description', 'Police_District', 'Resolution', 'Latitude', 'Longitude', 'Data_Source', 'Year', 'Month', 'DayOfWeek', 'DayName', 'Hour']


In [2]:
# Convert date to datetime for analysis
df['Incident_Date'] = pd.to_datetime(df['Incident_Date'])
df['Year'] = df['Incident_Date'].dt.year
df['Month'] = df['Incident_Date'].dt.month

print(f"Date range: {df['Incident_Date'].min()} to {df['Incident_Date'].max()}")

Date range: 2003-01-01 00:00:00 to 2025-12-31 00:00:00


In [3]:
# Identify key columns for duplicate detection
# Find the category column
category_cols = [col for col in df.columns if 'category' in col.lower() or 'unified' in col.lower()]
if 'Unified_Category' in df.columns:
    category_col = 'Unified_Category'
elif 'Category' in df.columns:
    category_col = 'Category'
else:
    category_col = category_cols[0] if category_cols else None

print(f"Using category column: {category_col}")

# Find district column
district_cols = [col for col in df.columns if 'district' in col.lower()]
district_col = 'Police_District' if 'Police_District' in df.columns else district_cols[0] if district_cols else None
print(f"Using district column: {district_col}")

# Find time column if exists
time_cols = [col for col in df.columns if 'time' in col.lower() and 'date' not in col.lower()]
time_col = time_cols[0] if time_cols else None
print(f"Using time column: {time_col}")

Using category column: Unified_Category
Using district column: Police_District
Using time column: Incident_Time


In [4]:
# Strategy 1: Check for exact duplicates across all columns
print("\n" + "="*60)
print("STRATEGY 1: Exact Duplicates (All Columns)")
print("="*60)

exact_duplicates = df.duplicated(keep=False)
n_exact = exact_duplicates.sum()
print(f"\nExact duplicate rows: {n_exact:,}")

if n_exact > 0:
    print(f"\nSample of exact duplicates:")
    print(df[exact_duplicates].head(10))


STRATEGY 1: Exact Duplicates (All Columns)

Exact duplicate rows: 0


In [5]:
# Strategy 2: Check for duplicates based on key fields
print("\n" + "="*60)
print("STRATEGY 2: Key Field Duplicates")
print("="*60)

# Define key columns for duplicate detection
key_columns = ['Incident_Date', category_col]
if district_col:
    key_columns.append(district_col)
if time_col:
    key_columns.append(time_col)

print(f"\nChecking duplicates based on: {key_columns}")

key_duplicates = df.duplicated(subset=key_columns, keep=False)
n_key = key_duplicates.sum()
print(f"Key field duplicates: {n_key:,}")

if n_key > 0:
    print(f"\nSample of key field duplicates:")
    sample_dups = df[key_duplicates].sort_values(key_columns).head(20)
    print(sample_dups[key_columns + ['Year']])


STRATEGY 2: Key Field Duplicates

Checking duplicates based on: ['Incident_Date', 'Unified_Category', 'Police_District', 'Incident_Time']
Key field duplicates: 131,675

Sample of key field duplicates:
        Incident_Date Unified_Category Police_District Incident_Time  Year
801785     2003-01-01          Assault       INGLESIDE         00:01  2003
1227652    2003-01-01          Assault       INGLESIDE         00:01  2003
856412     2003-01-01     Embezzlement        NORTHERN         12:00  2003
1309027    2003-01-01     Embezzlement        NORTHERN         12:00  2003
622135     2003-01-01     Embezzlement        SOUTHERN         12:00  2003
681342     2003-01-01     Embezzlement        SOUTHERN         12:00  2003
709975     2003-01-01            Fraud        SOUTHERN         00:01  2003
1125536    2003-01-01            Fraud        SOUTHERN         00:01  2003
1295687    2003-01-01            Fraud        SOUTHERN         00:01  2003
623560     2003-01-01            Fraud         T

In [6]:
# Strategy 3: Focus on 2018 overlap (May 2018)
print("\n" + "="*60)
print("STRATEGY 3: 2018 Overlap Analysis")
print("="*60)

# Check May 2018 specifically (likely overlap period)
may_2018 = df[(df['Year'] == 2018) & (df['Month'] == 5)]
print(f"\nMay 2018 records: {len(may_2018):,}")

# Check for duplicates in May 2018
may_2018_dups = may_2018.duplicated(subset=key_columns, keep=False)
print(f"May 2018 duplicates: {may_2018_dups.sum():,}")

# Check all of 2018
df_2018 = df[df['Year'] == 2018]
dups_2018 = df_2018.duplicated(subset=key_columns, keep=False)
print(f"\nTotal 2018 records: {len(df_2018):,}")
print(f"2018 duplicates: {dups_2018.sum():,}")
print(f"Percentage: {(dups_2018.sum() / len(df_2018) * 100):.2f}%")


STRATEGY 3: 2018 Overlap Analysis

May 2018 records: 5,874
May 2018 duplicates: 655

Total 2018 records: 69,673
2018 duplicates: 9,070
Percentage: 13.02%


In [7]:
# Analyze duplicate patterns by year
print("\n" + "="*60)
print("DUPLICATES BY YEAR")
print("="*60)

for year in sorted(df['Year'].unique()):
    year_data = df[df['Year'] == year]
    year_dups = year_data.duplicated(subset=key_columns, keep=False)
    n_dups = year_dups.sum()
    pct = (n_dups / len(year_data) * 100) if len(year_data) > 0 else 0
    
    if n_dups > 0:
        print(f"{year}: {n_dups:,} duplicates ({pct:.2f}% of {len(year_data):,} records)")


DUPLICATES BY YEAR
2003: 4,882 duplicates (8.65% of 56,438 records)
2004: 4,487 duplicates (8.01% of 56,038 records)
2005: 4,731 duplicates (8.21% of 57,612 records)
2006: 5,024 duplicates (8.79% of 57,129 records)
2007: 4,016 duplicates (7.62% of 52,717 records)
2008: 4,188 duplicates (7.91% of 52,921 records)
2009: 4,189 duplicates (8.49% of 49,342 records)
2010: 3,670 duplicates (7.90% of 46,481 records)
2011: 3,686 duplicates (7.72% of 47,725 records)
2012: 5,009 duplicates (9.07% of 55,233 records)
2013: 6,207 duplicates (10.34% of 60,051 records)
2014: 7,596 duplicates (12.20% of 62,276 records)
2015: 8,550 duplicates (12.66% of 67,511 records)
2016: 7,072 duplicates (11.09% of 63,778 records)
2017: 9,391 duplicates (13.37% of 70,239 records)
2018: 9,070 duplicates (13.02% of 69,673 records)
2019: 8,723 duplicates (12.93% of 67,453 records)
2020: 5,002 duplicates (9.46% of 52,899 records)
2021: 6,511 duplicates (10.91% of 59,660 records)
2022: 7,520 duplicates (11.54% of 65,162 

In [8]:
# Remove duplicates
print("\n" + "="*60)
print("REMOVING DUPLICATES")
print("="*60)

print(f"\nOriginal dataset: {len(df):,} records")

# First remove exact duplicates
df_clean = df.drop_duplicates()
print(f"After removing exact duplicates: {len(df_clean):,} records")
print(f"  Removed: {len(df) - len(df_clean):,} records")

# Then remove key field duplicates (keep first occurrence)
df_clean = df_clean.drop_duplicates(subset=key_columns, keep='first')
print(f"\nAfter removing key field duplicates: {len(df_clean):,} records")
print(f"  Removed: {len(df) - len(df_clean):,} records total")

# Calculate reduction
reduction = len(df) - len(df_clean)
reduction_pct = (reduction / len(df)) * 100
print(f"\n✓ Total reduction: {reduction:,} records ({reduction_pct:.2f}%)")


REMOVING DUPLICATES

Original dataset: 1,317,318 records
After removing exact duplicates: 1,317,318 records
  Removed: 0 records

After removing key field duplicates: 1,246,733 records
  Removed: 70,585 records total

✓ Total reduction: 70,585 records (5.36%)


In [ ]:

# Verify the cleaned data
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

# Check for remaining duplicates
remaining_dups = df_clean.duplicated(subset=key_columns, keep=False)
print(f"\nRemaining key field duplicates: {remaining_dups.sum():,}")

# Check year distribution
print(f"\nRecords per year (cleaned):")
year_counts = df_clean['Year'].value_counts().sort_index()
for year, count in year_counts.items():
    print(f"  {year}: {count:,}")

# Check 2018 specifically
df_2018_clean = df_clean[df_clean['Year'] == 2018]
print(f"\n2018 records after cleaning: {len(df_2018_clean):,}")
print(f"2018 monthly distribution:")
monthly_2018 = df_2018_clean.groupby('Month').size()
for month in range(1, 13):
    count = monthly_2018.get(month, 0)
    month_name = pd.to_datetime(f'2018-{month:02d}-01').strftime('%B')
    print(f"  {month_name:12s}: {count:,}")


VERIFICATION

Remaining key field duplicates: 0

Records per year (cleaned):
  2003: 53,878
  2004: 53,701
  2005: 55,128
  2006: 54,472
  2007: 50,620
  2008: 50,706
  2009: 47,140
  2010: 44,549
  2011: 45,786
  2012: 52,573
  2013: 56,759
  2014: 58,143
  2015: 62,857
  2016: 60,006
  2017: 65,096
  2018: 64,734
  2019: 62,662
  2020: 50,242
  2021: 56,153
  2022: 61,101
  2023: 58,809
  2024: 45,202
  2025: 36,416

2018 records after cleaning: 64,734
2018 monthly distribution:
  January     : 5,833
  February    : 4,877
  March       : 5,336
  April       : 5,290
  May         : 5,527
  June        : 5,337
  July        : 5,816
  August      : 5,726
  September   : 5,242
  October     : 5,529
  November    : 4,975
  December    : 5,246


In [10]:
# Save cleaned data
print("\n" + "="*60)
print("SAVING CLEANED DATA")
print("="*60)

# Drop the temporary Year and Month columns we added
df_clean = df_clean.drop(columns=['Year', 'Month'])

output_file = '../files/merged_crime_data_2003_2025_clean.csv'
df_clean.to_csv(output_file, index=False)
print(f"\n✓ Saved cleaned data to: {output_file}")
print(f"✓ Final record count: {len(df_clean):,}")

# Also create a backup of the original
backup_file = '../files/merged_crime_data_2003_2025_original_backup.csv'
print(f"\n✓ Original data backed up to: {backup_file}")
print("  (Run this manually if you want to keep a backup)")


SAVING CLEANED DATA

✓ Saved cleaned data to: ../files/merged_crime_data_2003_2025_clean.csv
✓ Final record count: 1,246,733

✓ Original data backed up to: ../files/merged_crime_data_2003_2025_original_backup.csv
  (Run this manually if you want to keep a backup)


## Summary

The duplicate removal process:

1. **Identified duplicates** based on key fields (date, category, district, time)
2. **Removed exact duplicates** first
3. **Removed key field duplicates** keeping the first occurrence
4. **Verified** the cleaned dataset has no remaining duplicates

### Next Steps:

1. Replace your original merged file with the cleaned version:
   ```python
   # In your terminal or a new cell:
   # mv ../files/merged_crime_data_2003_2025.csv ../files/merged_crime_data_2003_2025_original_backup.csv
   # mv ../files/merged_crime_data_2003_2025_clean.csv ../files/merged_crime_data_2003_2025.csv
   ```

2. Re-run the verification notebook to see the cleaned results

3. Update any analysis notebooks to use the cleaned data